# Trinidad, Colorado Distribution Models

This notebook documents the local-data workflow for building synthetic distribution feeder models for Trinidad, Colorado, USA. It produces one `DistributionSystem` JSON per feeder and an interactive HTML map for all substations.

Run from the SHIFT repository root in the `shift` Conda environment.

## Data sources and artifacts

| Data | Original online resource | Local artifact used here |
| --- | --- | --- |
| Address points | [Colorado Information Marketplace](https://data.colorado.gov/) public geospatial data portal; search for the Colorado Master Address Database | `/Users/alatif/Downloads/Master_Address_Public.gdb`, layer `LasAnimas`, filtered to ZIP `81082` |
| Roads | [OpenStreetMap](https://www.openstreetmap.org/) data, queried through [Overpass API](https://overpass-api.de/) | `data/trinidad/colorado.osm.pbf` |
| Substations | [OpenStreetMap](https://www.openstreetmap.org/) `power=substation` features, queried through [Overpass API](https://overpass-api.de/) | Retrieved during the build; any cache is local |
| Equipment catalog | `tests/models/p1rhs7_1247.json` in this repository | Same local catalog |

Users must obtain the address geodatabase and prepare an OpenStreetMap road PBF covering Trinidad before running this notebook. The `data/trinidad` and `outputs/trinidad_co` directories are local working directories and are not included in the repository.

In [ ]:
from collections import defaultdict, deque
import math
from pathlib import Path
import geopandas as gpd
import numpy as np
from shapely import box
from shapely.ops import unary_union

REPO = Path.cwd()
GDB_PATH = Path("/Users/alatif/Downloads/Master_Address_Public.gdb")
PBF_PATH = REPO / "data" / "trinidad" / "colorado.osm.pbf"
CATALOG_PATH = REPO / "tests" / "models" / "p1rhs7_1247.json"
EXPORT_PATH = REPO / "outputs" / "trinidad_co"
HTML_PATH = REPO / "examples" / "trinidad" / "trinidad_substations.html"
for path in (GDB_PATH, PBF_PATH, CATALOG_PATH):
    if not path.exists():
        raise FileNotFoundError(path)
print(f"Using repository: {REPO}")

## 1. Load address points and define the service area

The postal area is reduced to the dense connected urban block. Address points are used as parcel/load candidates; they are not claims about existing electrical assets.

In [ ]:
addresses = gpd.read_file(GDB_PATH, layer="LasAnimas")
trinidad = addresses[addresses["Zipcode"].astype(str) == "81082"].copy()
if trinidad.empty:
    raise ValueError("No address points found for ZIP code 81082")


def trinidad_service_polygon(points):
    cell = 0.01
    x_cells = np.floor(points.geometry.x.to_numpy() / cell).astype(int)
    y_cells = np.floor(points.geometry.y.to_numpy() / cell).astype(int)
    counts = defaultdict(int)
    for x_cell, y_cell in zip(x_cells, y_cells):
        counts[(x_cell, y_cell)] += 1
    dense = {key for key, count in counts.items() if count >= 20}
    seed = max(counts, key=counts.get)
    component, queue = {seed}, deque([seed])
    while queue:
        x_cell, y_cell = queue.popleft()
        for dx in (-1, 0, 1):
            for dy in (-1, 0, 1):
                neighbor = (x_cell + dx, y_cell + dy)
                if neighbor in dense and neighbor not in component:
                    component.add(neighbor)
                    queue.append(neighbor)
    cells = [box(x * cell, y * cell, (x + 1) * cell, (y + 1) * cell) for x, y in component]
    area = unary_union(cells).buffer(0.002)
    return max(area.geoms, key=lambda item: item.area) if hasattr(area, "geoms") else area


service_polygon = trinidad_service_polygon(trinidad)
latitude = float(np.median(trinidad.geometry.y.to_numpy()))
km_per_degree = 111.32 * math.cos(math.radians(latitude))
print(
    f"{len(trinidad):,} address points; {service_polygon.area * 111.32 * km_per_degree:.1f} km² service area"
)

## 2. Configure local inputs and build models

The feeder pipeline discovers substations, creates feeder cells, builds electrical networks, maps catalog equipment, and exports each result with `DistributionSystem.to_json` to `substation_<id>/feeder_<n>.json`. Point parcels use 25 address points as the transformer clustering target.

In [ ]:
from gdm.distribution import DistributionSystem
from shift.openstreet_roads import set_local_pbf
from shift.feeder_models import (
    ClusteringConfig,
    ExportConfig,
    FeederConfig,
    FeederModelConfig,
    ParcelSourceConfig,
    PRSGConfig,
    TransformerConfig,
    VoltageConfig,
    build_feeder_models,
)

EXPORT_PATH.mkdir(parents=True, exist_ok=True)
parcel_path = EXPORT_PATH / "trinidad_address_points.geojson"
trinidad.to_file(parcel_path, driver="GeoJSON")
set_local_pbf(str(PBF_PATH))
catalog = DistributionSystem.from_json(CATALOG_PATH)
config = FeederModelConfig(
    export=ExportConfig(folder=EXPORT_PATH),
    feeders=FeederConfig(phase_method="greedy"),
    parcels=ParcelSourceConfig(
        source="geodataframe",
        path=parcel_path,
        name_column="AddrFull",
        field_mapper="gis",
        column_map={"city": "PlaceName", "postal_address": "Zipcode"},
        local_pbf_path=PBF_PATH,
    ),
    clustering=ClusteringConfig(
        strategy="area_aware",
        target_area_per_transformer_m2=25.0,
        dedicated_transformer_area_m2=22000 * 0.09290304,
    ),
    prsg=PRSGConfig(
        offline=False,
        routing_strategy="SteinerTreeStrategy",
        secondary_strategy="DelaunayStrategy",
    ),
    transformers=TransformerConfig(type="THREE_PHASE", capacity_kva=500.0),
    voltages=VoltageConfig(
        primary_voltage_kv=12.47, secondary_voltage_kv=0.48, use_substation_voltage=False
    ),
)
results = build_feeder_models(service_polygon, config, catalog=catalog)
print(f"Built and exported {len(results)} feeder models")

In [ ]:
feeder_files = sorted(EXPORT_PATH.glob("substation_*/feeder_*.json"))
print(f"JSON exports: {len(feeder_files)}")
for path in feeder_files:
    print(path.relative_to(EXPORT_PATH))
validated = DistributionSystem.from_json(feeder_files[0])
print(f"Validated model: {validated.name}")

## 3. Plot all substations

The shared plotting helper loads every feeder recursively, calls `DistributionSystem.to_gdf()`, and creates a Plotly map with primary/secondary lines, buses, loads, a legend, and a feeder isolation dropdown.

In [ ]:
import sys

sys.path.insert(0, str(REPO / "scripts"))
from plot_trinidad_substations import build_figure, discover_feeder_files

systems = discover_feeder_files(EXPORT_PATH)
figure = build_figure(systems, root=EXPORT_PATH, map_type="scattermap")
figure.write_html(str(HTML_PATH), include_plotlyjs="cdn", full_html=True)
print(f"Wrote {len(systems)} feeders to {HTML_PATH}")